In [0]:
dbutils.widgets.text("catalog","sl")
catalog = dbutils.widgets.get("catalog")

#region

In [0]:
df = spark.table(f"{catalog}.bronze.region")

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("r_regionkey", trim("r_regionkey").cast("int")) \
    .withColumn("r_name",upper(trim("r_name"))) \
    .withColumn("r_comment", trim("r_comment"))

df = df.drop("ingested_at","source_path","pipeline_run_id")

duplicate_keys = df.groupBy("r_regionkey").count().filter(col("count") > 1).select("r_regionkey")

df = df.withColumn("validation_error", 
    when(col("r_regionkey").isNull(), "INVALID_REGION_ID")
    .otherwise(when(col("r_regionkey").isin([row["r_regionkey"] for row in duplicate_keys.collect()]), "DUPLICATE_REGION_ID")))

valid_df = df.filter(col("validation_error").isNull())
quarantine_df = df.filter(col("validation_error").isNotNull())

quarantine_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.region_quarantine")

valid_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.region")

In [0]:
valid_df.count()

In [0]:
quarantine_df.count()

#nation

In [0]:
df = spark.table(f"{catalog}.bronze.nation")

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("n_nationkey", trim("n_nationkey").cast("int")) \
    .withColumn("n_name", upper(trim("n_name"))) \
    .withColumn("n_regionkey", trim("n_regionkey").cast("int")) \
    .withColumn("n_comment", trim("n_comment"))

df = df.drop("ingested_at", "source_path","pipeline_run_id")

duplicate_keys = df.groupBy("n_nationkey").count().filter(col("count") > 1).select("n_nationkey")

valid_region_keys = [row["r_regionkey"] for row in spark.table(f"{catalog}.silver.region").select("r_regionkey").distinct().collect()]

df = df.withColumn("validation_error", 
    when(col("n_nationkey").isNull(), "INVALID_NATION_ID")
    .when(~col("n_regionkey").isin(valid_region_keys), "INVALID_FK_REGION")
    .when(col("n_nationkey").isin([row["n_nationkey"] for row in duplicate_keys.collect()]), "DUPLICATE_NATION_ID")
    .otherwise(lit(None).cast("string")))

valid_df = df.filter(col("validation_error").isNull())
quarantine_df = df.filter(col("validation_error").isNotNull())

quarantine_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.nation_quarantine")

valid_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.nation")

In [0]:
valid_df.count()

In [0]:
quarantine_df.count()

#airline

In [0]:
df = spark.table(f"{catalog}.bronze.airline")

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("al_airlineid",trim("al_airlineid").cast("int")) \
    .withColumn("al_name", trim("al_name")) \
    .withColumn("al_address",trim("al_address")) \
    .withColumn("al_rating",col("al_rating").cast("decimal(10,2)")) \
    .withColumn("al_comment",trim("al_comment")) \
    .withColumn("al_nationkey",trim("al_nationkey").cast("int"))

df = df.drop("ingested_at", "source_path","pipeline_run_id")

valid_nation_keys = [row["n_nationkey"] for row in spark.table(f"{catalog}.silver.nation").select("n_nationkey").distinct().collect()]

df = df.withColumn("validation_error",
           when(col("al_airlineid").isNull(),"NULL_PK")  
           .when(~col("al_nationkey").isin(valid_nation_keys),"INVALID_FK_NATION")                                                               
           .otherwise(""))

valid_df = df.filter(col("validation_error") == "")
quarantine_df = df.filter(col("validation_error") != "")


quarantine_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.airline_quarantine")

valid_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.airline")

In [0]:
valid_df.count()

In [0]:
quarantine_df.count()

#route

In [0]:
df = spark.table(f"{catalog}.bronze.route")

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("ro_routeid",col("ro_routeid").cast("int")) \
     .withColumn("ro_name",upper(trim("ro_name"))) \
    .withColumn("ro_aircraft", trim("ro_aircraft")) \
    .withColumn("ro_airline_brand", trim("ro_airline_brand")) \
    .withColumn("ro_routetype", upper(trim("ro_routetype"))) \
    .withColumn("ro_capacity",col("ro_capacity").cast("int")) \
     .withColumn("ro_baseprice",col("ro_baseprice").cast("decimal(10,2)")) \
    .withColumn("ro_cabinconfig",upper(trim("ro_cabinconfig"))) \
    .withColumn("ro_comment",trim("ro_comment"))
    
df = df.drop("ingested_at", "source_path","pipeline_run_id")

from pyspark.sql.window import Window

w = Window.partitionBy("ro_routeid").orderBy(lit(1))
df = df.withColumn("dup_row_num", row_number().over(w))

df = df.withColumn("validation_error", 
    when(col("ro_routeid").isNull(), "INVALID_PK")
    .when(col("ro_baseprice") <= 0 , "INVALID BASEPRICE")
    .when(col("dup_row_num") > 1, "DUPLICATE_ROUTE_ID")
    .otherwise(lit(None).cast("string")))

df = df.drop("dup_row_num")

valid_df = df.filter(col("validation_error").isNull())
quarantine_df = df.filter(col("validation_error").isNotNull())

quarantine_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.route_quarantine")

valid_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.route")



In [0]:
valid_df.count()

In [0]:
quarantine_df.count()

#airline_route

In [0]:
df = spark.table(f"{catalog}.bronze.airline_route")

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("ar_airlineid",col("ar_airlineid").cast("int")) \
    .withColumn("ar_routeid",col("ar_routeid").cast("int")) \
    .withColumn("ar_available_seats",col("ar_available_seats").cast("int")) \
    .withColumn("ar_operatingcost",col("ar_operatingcost").cast("decimal(10,2)")) \
    .withColumn("ar_comment",trim("ar_comment"))

df = df.drop("source_path","pipeline_run_id","ingested_at")

valid_airline_ids = [row["al_airlineid"] for row in spark.table(f"{catalog}.silver.airline").select("al_airlineid").distinct().collect()]
valid_route_ids = [row["ro_routeid"] for row in spark.table(f"{catalog}.silver.route").select("ro_routeid").distinct().collect()]

# Keep 1 of each duplicate composite PK in valid; rest go to quarantine
from pyspark.sql.window import Window

w = Window.partitionBy("ar_airlineid","ar_routeid").orderBy(lit(1))
df = df.withColumn("dup_row_num", row_number().over(w))


df = df.withColumn("validation_error", concat_ws(",", when((col("ar_airlineid").isNull() | col("ar_routeid").isNull()), "NULL_PK"),
    when(~col("ar_airlineid").isin(valid_airline_ids), "INVALID_FK_AIRLINE"),
    when(~col("ar_routeid").isin(valid_route_ids), "INVALID_FK_ROUTE"),
    when(col("ar_operatingcost") < 0, "INVALID_OPERATING_COST"),
    when(col("dup_row_num") > 1, "DUPLICATE_PK"),))

df = df.drop("dup_row_num")

valid_df = df.filter(col("validation_error") == "")
quarantine_df = df.filter(col("validation_error") != "")


quarantine_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.airline_route_quarantine")

valid_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.airline_route")


In [0]:
valid_df.count()

In [0]:
quarantine_df.count()

#passengers

In [0]:
df = spark.table(f"{catalog}.bronze.passenger")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


df = df.withColumn("pa_passengerid",col("pa_passengerid").cast("int")) \
    .withColumn("pa_name",trim("pa_name")) \
    .withColumn("pa_address",trim("pa_address")) \
    .withColumn("pa_nationkey",trim("pa_nationkey").cast("int")) \
    .withColumn("pa_loyaltypoints",col("pa_loyaltypoints").cast("decimal(10,2)")) \
    .withColumn("pa_travelpurpose",upper(trim("pa_travelpurpose"))) \
    .withColumn("pa_comment",trim("pa_comment"))

df = df.drop("source_path","pipeline_run_id","ingested_at")

valid_nation_keys = [row["n_nationkey"] for row in spark.table(f"{catalog}.silver.nation").select("n_nationkey").distinct().collect()]

w = Window.partitionBy("pa_passengerid").orderBy(lit(1))
df = df.withColumn("dup_row_num", row_number().over(w))

df = df.withColumn("validation_error", 
    when(col("pa_passengerid").isNull(), "INVALID_PK")
    .when(~col("pa_nationkey").isin(valid_nation_keys),"INVALID_FK_NATION")
    .when(col("dup_row_num") > 1, "DUPLICATE_PASSENGER_ID")
    .when(col("pa_name").isNull(),"NULL pa_name")
    .otherwise(lit(None).cast("string")))

df = df.drop("dup_row_num")

valid_df = df.filter(col("validation_error").isNull())
quarantine_df = df.filter(col("validation_error").isNotNull())

quarantine_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.passengers_quarantine")

valid_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.passengers")



In [0]:
valid_df.count()

In [0]:
quarantine_df.count()

#bookings

In [0]:
df = spark.table(f"{catalog}.bronze.booking")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


df = df.withColumn("bk_bookingid",col("bk_bookingid").cast("int")) \
    .withColumn("bk_passengerid",col("bk_passengerid").cast("int")) \
    .withColumn("bk_status",upper(trim("bk_status"))) \
    .withColumn("bk_totalprice",col("bk_totalprice").cast("decimal(10,2)")) \
    .withColumn("bk_bookingdate",to_date("bk_bookingdate","yyyy-MM-dd")) \
    .withColumn("bk_flightdate",to_date("bk_flightdate","yyyy-MM-dd")) \
    .withColumn("bk_bookingpriority",trim("bk_bookingpriority")) \
    .withColumn("bk_agent",trim("bk_agent")) \
    .withColumn("bk_comment",trim("bk_comment"))

df = df.drop("source_path","pipeline_run_id","ingested_at")

valid_passengers_keys = [row["pa_passengerid"] for row in spark.table(f"{catalog}.silver.passengers").select("pa_passengerid").distinct().collect()]

w = Window.partitionBy("bk_bookingid").orderBy(lit(1))
df = df.withColumn("dup_row_num", row_number().over(w))

df = df.withColumn("validation_error", 
    when(col("bk_bookingid").isNull(), "INVALID_PK")
    .when(~col("bk_passengerid").isin(valid_passengers_keys),"INVALID_FK_PASSENGER")
    .when(col("dup_row_num") > 1, "DUPLICATE_BOOKING_ID")
    .when(col("bk_flightdate") < col("bk_bookingdate"),"INVALID_DATE")
    .when(col("bk_totalprice") <= 0 , "INVALID_TOTAL_PRICE")
    .otherwise(lit(None).cast("string")))

df = df.drop("dup_row_num")

valid_df = df.filter(col("validation_error").isNull())
quarantine_df = df.filter(col("validation_error").isNotNull())

quarantine_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.booking_quarantine")

valid_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.booking")



In [0]:
valid_df.count()

In [0]:
quarantine_df.count()

#flight_booking

In [0]:
df = spark.table(f"{catalog}.bronze.flight_booking")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


df = df.withColumn("fb_bookingid",col("fb_bookingid").cast("int")) \
    .withColumn("fb_routeid",col("fb_routeid").cast("int")) \
    .withColumn("fb_airlineid",col("fb_airlineid").cast("int")) \
    .withColumn("fb_seats",col("fb_seats").cast("int")) \
    .withColumn("fb_ticketprice",col("fb_ticketprice").cast("decimal(10,2)")) \
    .withColumn("fb_discount",col("fb_discount").cast("decimal(10,2)")) \
    .withColumn("fb_comment",trim("fb_comment"))

df = df.drop("source_path","pipeline_run_id","ingested_at","_rescued_data")

valid_booking_ids = [row["bk_bookingid"] for row in spark.table(f"{catalog}.silver.booking").select("bk_bookingid").distinct().collect()]

valid_route_ids = [row["ro_routeid"] for row in spark.table(f"{catalog}.silver.route").select("ro_routeid").distinct().collect()]

valid_airline_ids = [row["al_airlineid"] for row in spark.table(f"{catalog}.silver.airline").select("al_airlineid").distinct().collect()]

# Keep 1 of each duplicate composite PK in valid; rest go to quarantine

w = Window.partitionBy("fb_bookingid","fb_linenumber").orderBy(lit(1))
df = df.withColumn("dup_row_num", row_number().over(w))


df = df.withColumn("validation_error", concat_ws(",", when((col("fb_bookingid").isNull() | col("fb_linenumber").isNull()), "NULL_PK"),
    when(~col("fb_bookingid").isin(valid_booking_ids), "INVALID_FK_BOOKING"),
    when((col("fb_routeid").isNull() |  ~col("fb_routeid").isin(valid_route_ids)), "INVALID_FK_ROUTE"),
    when(~col("fb_airlineid").isin(valid_airline_ids), "INVALID_FK_AIRLINE"),
    when(col("fb_ticketprice") <= 0, "INVALID_TICKET_PRICE"),
    when((col("fb_discount") < 0) | (col("fb_discount") > 1), "INVALID_DISCOUNT_PRICE"),
    when(col("fb_departuredate") > col("fb_arrivaldate"),"INVALID_DATE"),
    when(col("fb_seats") < 1 ,"INVALID_SEATS"),
   
    ))

df = df.drop("dup_row_num")

valid_df = df.filter(col("validation_error") == "")
quarantine_df = df.filter(col("validation_error") != "")


quarantine_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.flight_booking_quarantine")

valid_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.flight_booking")


In [0]:
valid_df.count()

In [0]:
quarantine_df.count()